# Homework 6: BERT & Contextual Embeddings
## CIS 5300: Natural Language Processing
## Total Points: 100

---

### Overview

In this assignment, you will explore how to use **BERT** (Bidirectional Encoder Representations from Transformers) for various NLP tasks. BERT is a pretrained bidirectional transformer encoder that produces rich **contextual embeddings**, representations of words that depend on the surrounding context.

You will:
1. **Use BERT zero-shot** (without any fine-tuning) for sentiment analysis via mask-filling
2. **Fine-tune BERT** for 5-class sentiment analysis on Yelp reviews
3. **Fine-tune BERT** for Natural Language Inference (NLI) on MNLI
4. **Fine-tune BERT** for Named Entity Recognition (NER) on CoNLL-2003
5. **Analyze** what pre-training buys you, where BERT fails, and how it handles adversarial inputs
6. **Reflect** on transfer learning, contextual embeddings, and the pretrain-finetune paradigm

### Learning Goals

After completing this homework, you will be able to:

1. **Use pretrained BERT** for zero-shot tasks and understand its limitations *(Module 8, Lecture 1.2)*
2. **Fine-tune for sequence classification** using the [CLS] token *(Module 8, Lecture 2.2; Chapter 10, §10.4.1)*
3. **Fine-tune for sentence-pair classification** using [SEP] *(Module 8, Lecture 2.2; Chapter 10, §10.4.2)*
4. **Fine-tune for token classification** with BIO tagging and subword alignment *(Module 8, Lecture 2.3; Chapter 10, §10.5)*
5. **Evaluate transfer learning** through ablation and adversarial testing
6. **Navigate the HuggingFace ecosystem** (Transformers, Datasets, Trainer)

### Point Breakdown

| Section | Points | Type |
|---------|--------|------|
| 1. Zero-Shot BERT | 15 | Code (autograded) |
| 2. Fine-Tune for Sentiment | 20 | Code (autograded) |
| 3. Fine-Tune for NLI | 25 | Code (autograded) + manually graded |
| 4. Fine-Tune for NER | 25 | Code (autograded) |
| 5. Analysis & Ablation | 15 | Code + Writeup |
| **Total** | **100** | |

### Deliverables

| File | Description |
|------|-------------|
| `hw06_notebook.ipynb` | This notebook with all code cells completed and run |
| `hw06.py` | File → Download → Download .py                  |
| `hw06_writeup.pdf` | Written answers to all [WRITEUP] questions |
| `eval_results_sentiment.json` | Auto-saved when you ran the sentiment eval cell |
| `eval_results_nli.json` | Auto-saved when you ran the NLI eval cell |
| `eval_results_ner.json` | Auto-saved when you ran the NER eval cell |

### Recommended Reading

- [Chapter 10: Masked Language Models](https://web.stanford.edu/~jurafsky/slp3/10.pdf), Jurafsky & Martin, SLP 3rd Edition
- [BERT: Pre-training of Deep Bidirectional Transformers](https://arxiv.org/abs/1810.04805), Devlin et al., NAACL 2019
- Module 8 Lectures: M07.1.1 through M07.2.3

### Which Sections Need Which Resources

- **Sections 1-4** are independent of each other: each loads its own dataset and trains its own model. You can run them in any order, and a bug in one will not block the others.
- **Section 5** depends on the trained Yelp model from Section 2 (5.1 and 5.2).

### GPU Requirement

Fine-tuning BERT requires a GPU. If using Google Colab, go to **Runtime > Change Runtime Type** and set **Hardware Accelerator** to **GPU**. Training on CPU will be ~10x slower.

**Heads up about Colab credits.** Free Colab gives you a limited amount of GPU time per day, and the four fine-tuning runs in this homework will use a meaningful chunk of it. If you run out partway through, two options:

- Wait ~24 hours for the free tier to reset, then resume.
- Sign up for a free [Colab Pro for Students](https://colab.research.google.com/signup) trial with your Penn email — this gives you noticeably more GPU time and is enough for this homework.

You can also save a trained model to Drive (`trainer.save_model("/content/drive/MyDrive/...")`) so a session timeout doesn't cost you a re-train.

---
## Section 0: Setup

Run the following cells to install dependencies and configure the environment. **Do not modify these cells.**

In [ ]:
%%capture
# Install dependencies. Pin transformers / datasets to a combination known to
# work with `evaluate` — the unpinned latest sometimes drops symbols that
# `evaluate` imports (e.g. `transformers.TFPreTrainedModel`).
!pip install -q -U huggingface_hub fsspec
!pip install -q "transformers==4.57.6"
!pip install -q --upgrade accelerate
!pip install -q evaluate
!pip install -q "datasets==3.6.0"
!pip install -q seqeval scikit-learn

# Quiet HuggingFace's chatty deprecation warnings so you can read your output.
import warnings, logging, os
warnings.filterwarnings("ignore")
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("datasets").setLevel(logging.ERROR)


In [ ]:
# ── Gradescope result loaders ────────────────────────────────
# These functions let the autograder read your saved results.
# Do NOT modify them.
import json as _json, os as _os

def eval_results_sentiment():
    try:
        with open('eval_results_sentiment.json') as _f: return _json.load(_f)
    except FileNotFoundError:
        return None

def eval_results_nli():
    try:
        with open('eval_results_nli.json') as _f: return _json.load(_f)
    except FileNotFoundError:
        return None

def eval_results_ner():
    try:
        with open('eval_results_ner.json') as _f: return _json.load(_f)
    except FileNotFoundError:
        return None

In [ ]:
########## DO NOT MODIFY ##########
import os
os.environ["WANDB_DISABLED"] = "true"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import numpy as np
import torch
from transformers import (
    pipeline,
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModelForTokenClassification,
    Trainer,
    TrainingArguments,
)
from datasets import load_dataset

# Detect GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device_id = 0 if torch.cuda.is_available() else -1
print(f"Using device: {device}")
if not torch.cuda.is_available():
    print("WARNING: No GPU detected. Fine-tuning will be very slow.")
    print("If using Colab: Runtime > Change Runtime Type > GPU")

---
## Section 1: Zero-Shot Application of BERT (15 points)

BERT was trained via **Masked Language Modeling** (MLM) — it learns to predict masked tokens in a sequence. Can we exploit this ability to do sentiment analysis *without any fine-tuning*?

The idea: append a template like `"I give it a score of [MASK] out of 5."` to each review, and ask BERT which token (1-5) is most likely to fill the [MASK]. This is a **zero-shot** approach — we use the model as-is, without any task-specific training.

**Connection to Lecture M07.1.2:** BERT's MLM objective trains it to predict missing tokens given bidirectional context. Here we test whether this ability transfers to sentiment classification.

In [ ]:
# Load BERT for mask-filling
bert_model = pipeline("fill-mask", model="bert-base-cased", device=device_id)

if __name__ == '__main__':
    # Demo: BERT predicts 'Italian' with 84.7% probability
    print(bert_model("I went to an [MASK] restaurant and ordered pasta."))


In [ ]:
if __name__ == '__main__':
    # We can use the `targets` parameter to restrict predictions to specific tokens
    # NOTE: targets must be STRINGS, not integers
    result = bert_model(
        "I just saw a movie today and it was really great. My opinion of the movie is [MASK].",
        targets=["positive", "negative"]
    )
    print(result)
    # BERT assigns higher probability to "positive" — correct!


### Load the Yelp Reviews Dataset

We'll use the [Yelp Reviews Full](https://huggingface.co/datasets/yelp_review_full) dataset: 650,000 reviews with star ratings from 1 to 5. The label is `0` = 1 star, `1` = 2 stars, ..., `4` = 5 stars.

In [ ]:
if __name__ == '__main__':
    dataset = load_dataset("Yelp/yelp_review_full")
    print(dataset)


### Problem 1.1: Concatenate the Mask Template (4 points) [autograded]

Write a function that appends the template `" I give it a score of [MASK] out of 5."` to each review's `text` field and stores the result in a new field called `input`.

Use the HuggingFace [Dataset.map()](https://huggingface.co/docs/datasets/process#map) function to apply your function to every row.

In [ ]:
def concatenate_mask(row):
    """Append the mask-filling template to the text.

    Args:
        row: dict with key "text" (str)

    Returns:
        dict: the input row with a new key "input" containing
              the original text followed by the template string.
    """
    template = " I give it a score of [MASK] out of 5."
    ## YOUR CODE HERE
    return row

if __name__ == '__main__':
    test_data = dataset["test"].map(concatenate_mask)
    print(test_data[0]["input"][:200])  # Preview

# Sanity checks — 1.1
_test = {"text": "Great food!", "label": 4}
_r = concatenate_mask(_test)
assert "input" in _r, "concatenate_mask must add an 'input' key"
assert "[MASK]" in _r["input"], "input must contain [MASK]"
assert _r["input"].startswith("Great food!"), "input must start with the original text"
assert _r.get("text") == "Great food!", "original 'text' field must be unchanged"
print("✓ 1.1 sanity checks passed")


### Problem 1.2: Filter for BERT's Max Length (4 points) [autograded]

BERT has a maximum context window of **512 tokens**. If an input exceeds this, BERT will raise an error. Use the [Dataset.filter()](https://huggingface.co/docs/datasets/process#select-and-filter) function to keep only examples whose tokenized `input` has 512 or fewer tokens.

**Hint:** Use `tokenizer(row["input"])` to tokenize, then check `len(result["input_ids"])`. See the [AutoTokenizer docs](https://huggingface.co/docs/transformers/model_doc/auto#transformers.AutoTokenizer) for details.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

def filter_for_max_length(row):
    """Return True if the tokenized input fits within BERT's max length.

    Args:
        row: dict with key "input" (str)

    Returns:
        bool: True if len(tokenized input) <= 512, False otherwise
    """
    ## YOUR CODE HERE
    return True  # Replace this

if __name__ == '__main__':
    filtered_data = test_data.filter(filter_for_max_length)
    print(f"Kept {len(filtered_data)} / {len(test_data)} examples (filtered {len(test_data) - len(filtered_data)})")

# Sanity checks — 1.2
assert filter_for_max_length({"input": "Short."}) == True, "short input should pass"
assert filter_for_max_length({"input": "word " * 600}) == False, "600-word input exceeds 512 tokens"
print("✓ 1.2 sanity checks passed")


In [ ]:
if __name__ == '__main__':
    # Sample 100 examples for efficiency (do not change)
    sampled_data = filtered_data.shuffle(seed=42).select(range(100))


### Problem 1.3: Apply BERT to Each Example (7 points) [autograded]

Write a function that runs BERT on each example and returns the most likely star rating (1-5) as an integer.

**Important details:**
- Use `targets=["1", "2", "3", "4", "5"]` — targets must be **strings**, not integers, because BERT predicts tokens
- The pipeline returns a list of dicts sorted by probability. The first element has the highest probability.
- Extract the `token_str` from the top prediction and convert it to `int`
- Store the result as `row["score"]`

In [ ]:
def apply_bert(row):
    """Use BERT mask-filling to predict a star rating for the review.

    Use the `bert_model` pipeline loaded above with `targets=["1","2","3","4","5"]`
    so it only scores the digits 1-5 at the [MASK] position. Take the top-ranked
    prediction and convert its `token_str` to an integer.

    Args:
        row: dict with key "input" (str containing [MASK])

    Returns:
        dict: {"score": int}, the predicted star rating in 1..5.
    """
    ## YOUR CODE HERE
    return {"score": None}

if __name__ == '__main__':
    predicted_data = sampled_data.map(apply_bert)
    print(f"Sample predictions: {predicted_data['score'][:10]}")
    print(f"Sample true labels: {[l+1 for l in predicted_data['label'][:10]]}")


In [ ]:
if __name__ == '__main__':
    # Evaluate zero-shot accuracy
    # Labels in the dataset are 0-indexed (0=1 star, 4=5 stars), so we add 1
    correct = sum(s == l + 1 for s, l in zip(predicted_data["score"], predicted_data["label"]))
    accuracy = correct / len(predicted_data)
    print(f"Zero-shot accuracy: {accuracy:.2%}")
    print(f"\nFor reference, random guessing on 5 classes = 20%.")
    print(f"BERT barely beats random chance! This motivates the need for fine-tuning.")


---
## Section 2: Fine-Tuning BERT for Sentiment Analysis (20 points)

Now let's **fine-tune** BERT for the same sentiment analysis task. Instead of using the mask-filling head, we'll replace it with a **classification head** — a linear layer that maps the [CLS] token's output to 5 class probabilities.

**Connection to Lecture M07.2.2:** The [CLS] token's output vector serves as a "sentence embedding" that captures the meaning of the entire input. The classification head learns a mapping from this embedding to class labels.

### What is fine-tuning?

Fine-tuning involves:
1. **Loading a pretrained model** (BERT with its learned weights)
2. **Replacing the head** (swap MLM head for a classification head)
3. **Training on labeled data** to learn the new head's weights (and optionally adjust BERT's weights)

This is **transfer learning**: knowledge from pretraining (understanding language) transfers to the downstream task (sentiment classification), requiring far less labeled data than training from scratch.

In [ ]:
if __name__ == '__main__':
    # Load BERT with a classification head for 5 classes
    # AutoModelForSequenceClassification adds a linear layer on top of the [CLS] output
    classification_head_bert = AutoModelForSequenceClassification.from_pretrained(
        "bert-base-cased", num_labels=5
    )


### Problem 2.1: Tokenize the Dataset (5 points) [autograded]

Write a function that tokenizes each example's `text` field using the BERT tokenizer. You must:
- **Pad** short sequences to 512 tokens using `padding="max_length"`
- **Truncate** long sequences to 512 tokens using `truncation=True`

The tokenizer returns a dict with keys `input_ids`, `attention_mask`, and `token_type_ids`, all of length 512.

**Docs:** [PreTrainedTokenizer.__call__](https://huggingface.co/docs/transformers/main_classes/tokenizer#transformers.PreTrainedTokenizer.__call__)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

def tokenize_function(row):
    """Tokenize the text field with padding and truncation.

    Apply the tokenizer to `row["text"]` and return its output dict so the
    mapped dataset gains `input_ids`, `attention_mask`, and `token_type_ids`
    (each a list of 512 ints). See `PreTrainedTokenizer.__call__` for the
    `padding` and `truncation` arguments you need.

    Args:
        row: dict with key "text" (str)

    Returns:
        dict: tokenizer output (input_ids, attention_mask, token_type_ids).
    """
    ## YOUR CODE HERE
    return row

# Subsample for training efficiency (do not change)

if __name__ == '__main__':
    dataset = load_dataset("Yelp/yelp_review_full")
    train_data = dataset["train"].shuffle(seed=42).select(range(1000)).map(tokenize_function)
    eval_data = dataset["test"].shuffle(seed=42).select(range(1000)).map(tokenize_function)
    print(f"Train: {len(train_data)}, Eval: {len(eval_data)}")
    print(f"input_ids length: {len(train_data[0]['input_ids'])}")

# Sanity checks — 2.1
_r = tokenize_function({"text": "This is a test sentence."})
assert "input_ids" in _r, "tokenize_function must return 'input_ids'"
assert "attention_mask" in _r, "tokenize_function must return 'attention_mask'"
assert len(_r["input_ids"]) == 512, f"input_ids must be length 512, got {len(_r['input_ids'])}"
print("✓ 2.1 sanity checks passed")


### Problem 2.2: Define compute_metrics (10 points) [autograded]

The HuggingFace Trainer calls this function after each evaluation epoch. It receives `eval_pred`, a tuple of `(logits, labels)` where:
- `logits`: numpy array of shape `(n_examples, n_classes)` — raw model outputs
- `labels`: numpy array of shape `(n_examples,)` — true class labels

You should:
1. Convert logits to predicted class labels using `np.argmax`
2. Compute accuracy as the fraction of correct predictions
3. Return `{"accuracy": <float>}`

In [ ]:
def compute_metrics(eval_pred):
    """Compute accuracy from model logits and true labels.

    Args:
        eval_pred: tuple (logits, labels)
            logits: np.ndarray of shape (n, num_classes)
            labels: np.ndarray of shape (n,)

    Returns:
        dict: {"accuracy": float in [0, 1]}.
    """
    logits, labels = eval_pred
    ## YOUR CODE HERE
    return {"accuracy": None}

# Sanity checks — 2.2
import numpy as np
_logits = np.array([[0.9, 0.05, 0.02, 0.02, 0.01], [0.01, 0.02, 0.9, 0.05, 0.02]])
_labels = np.array([0, 2])
_r = compute_metrics((_logits, _labels))
assert "accuracy" in _r, "compute_metrics must return dict with 'accuracy' key"
assert abs(_r["accuracy"] - 1.0) < 1e-6, f"perfect predictions → accuracy=1.0, got {_r['accuracy']}"
_logits2 = np.array([[0.9, 0.1, 0.0, 0.0, 0.0], [0.9, 0.1, 0.0, 0.0, 0.0]])
_labels2 = np.array([0, 1])
_r2 = compute_metrics((_logits2, _labels2))
assert abs(_r2["accuracy"] - 0.5) < 1e-6, f"1/2 correct → accuracy=0.5, got {_r2['accuracy']}"
print("✓ 2.2 sanity checks passed")


### Problem 2.3: Train and Evaluate (5 points) [autograded]

The `Trainer` handles the training loop, evaluation, and checkpointing. Run the cells below to train BERT for 3 epochs. This should take ~5 minutes on GPU.

Your model should achieve at least **45% accuracy** on the eval set to receive full credit. With default hyperparameters, expect ~55-60%.

In [ ]:
if __name__ == '__main__':
    ########## DO NOT MODIFY ##########
    training_args = TrainingArguments(
        output_dir="yelp-training",
        eval_strategy="epoch",
        logging_strategy="epoch",
        learning_rate=5e-05,
        num_train_epochs=3,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        report_to="none",
        seed=42,
    )

    trainer = Trainer(
        model=classification_head_bert,
        args=training_args,
        train_dataset=train_data,
        eval_dataset=eval_data,
        compute_metrics=compute_metrics,
    )


In [ ]:
# Skip when imported as a module (slow training — ~5 min on GPU)
if __name__ == '__main__':
    # Train the model (~5 min on GPU)
    trainer.train()
    trainer.save_model("./yelpBERT")

In [ ]:
if __name__ == '__main__':
    # Evaluate: load the trained model into a pipeline and test
    yelpBERT = pipeline("text-classification", model="./yelpBERT", tokenizer=tokenizer, device=device_id)

    # Sanity check
    print(yelpBERT(["This place was amazing!",
                    "This place was OK.",
                    "This place was terrible."]))


In [ ]:
import json as _json

# Load saved eval results if available (populated during training below)
eval_results_sentiment = None
try:
    eval_results_sentiment = _json.load(open('eval_results_sentiment.json'))
    print(f"Sentiment accuracy: {eval_results_sentiment['eval_accuracy']:.2%}")
except (FileNotFoundError, KeyError):
    pass

if __name__ == '__main__':
    eval_results_sentiment = trainer.evaluate()
    _json.dump({'accuracy': eval_results_sentiment['eval_accuracy']}, open('eval_results_sentiment.json', 'w'))
    sentiment_eval_results = eval_results_sentiment  # alias for Section 5.2
    print(f"Sentiment accuracy: {eval_results_sentiment['eval_accuracy']:.2%}")
    assert eval_results_sentiment["eval_accuracy"] >= 0.45, (
        f"Accuracy should be >= 45%, got {eval_results_sentiment['eval_accuracy']:.2%}. "
        f"Check your tokenize_function and compute_metrics."
    )


> **📥 Download for Gradescope!**
> After running the cells above, download **eval_results_sentiment.json** from the Colab file browser (folder icon on the left sidebar) and include it in your Gradescope submission.

---
## Section 3: Fine-Tuning BERT for Natural Language Inference (25 points)

**Natural Language Inference** (NLI) is the task of determining the logical relationship between two sentences:
- **Entailment**: the premise logically implies the hypothesis
- **Neutral**: neither entailment nor contradiction
- **Contradiction**: the premise contradicts the hypothesis

This is a **sentence-pair classification** task. We encode both sentences into BERT by separating them with the special `[SEP]` token.

**Connection to Lecture M07.2.2 and Chapter 10, §10.4.2:** BERT was pretrained with Next Sentence Prediction (NSP), which trains the [CLS] token to represent relationships between sentence pairs. Fine-tuning for NLI leverages this ability.

In [ ]:
if __name__ == '__main__':
    # Load BERT with a 3-class classification head
    # Labels: 0 = entailment, 1 = neutral, 2 = contradiction
    classification_head_bert = AutoModelForSequenceClassification.from_pretrained(
        "bert-base-cased", num_labels=3
    )


In [ ]:
if __name__ == '__main__':
    # Load the Multi-Genre NLI dataset
    mnli = load_dataset("nyu-mll/multi_nli")
    print(mnli)
    print(f"\nExample:")
    print(f"  Premise:    {mnli['train'][0]['premise']}")
    print(f"  Hypothesis: {mnli['train'][0]['hypothesis']}")
    print(f"  Label:      {mnli['train'][0]['label']}")


### Problem 3.1: Filter Unlabeled Examples (4 points) [autograded]

Some MNLI examples have `label = -1`, meaning no annotator consensus was reached. Filter these out.

In [ ]:
def is_labeled(row):
    """Return True if the example has a valid label (not -1).

    Args:
        row: dict with key "label" (int)
    Returns:
        bool
    """
    ## YOUR CODE HERE
    return True  # Replace this

if __name__ == '__main__':
    mnli = mnli.filter(is_labeled)
    print(f"After filtering: {mnli}")

# Sanity checks — 3.1
assert is_labeled({"label": 0}) == True,  "label=0 (entailment) should be kept"
assert is_labeled({"label": 2}) == True,  "label=2 (contradiction) should be kept"
assert is_labeled({"label": -1}) == False, "label=-1 (no consensus) should be filtered"
print("✓ 3.1 sanity checks passed")


### Problem 3.2: Concatenate Premise and Hypothesis (4 points) [autograded]

BERT encodes sentence pairs by joining them with the `[SEP]` token. Create a new field `concat` with the format:

```
<premise> [SEP] <hypothesis>
```

**Note:** In practice, you could pass both sentences directly to the tokenizer as `tokenizer(premise, hypothesis)`, which handles [SEP] insertion automatically. Here we do it manually for pedagogical clarity.

In [ ]:
def concatenate(row):
    """Concatenate premise and hypothesis with [SEP].

    Args:
        row: dict with keys "premise" (str) and "hypothesis" (str)

    Returns:
        dict: input row with new key "concat" set to a string of the form
        "<premise> [SEP] <hypothesis>".
    """
    ## YOUR CODE HERE
    return row

if __name__ == '__main__':
    mnli = mnli.map(concatenate)
    print(f"Concatenated: {mnli['train'][0]['concat'][:200]}")

# Sanity checks — 3.2
_test = {"premise": "The cat sat.", "hypothesis": "An animal sat."}
_r = concatenate(_test)
assert "concat" in _r, "concatenate must add a 'concat' key"
assert "[SEP]" in _r["concat"], "concat must contain [SEP]"
assert _r["concat"].startswith("The cat sat."), "premise must come first"
assert _r["concat"].endswith("An animal sat."), "hypothesis must come last"
print("✓ 3.2 sanity checks passed")


### Problem 3.3: Tokenize the Concatenated Input (4 points) [manually graded]

Tokenize the `concat` field with padding and truncation, just as in Section 2.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

def tokenize_function_nli(row):
    """Tokenize the concatenated premise+hypothesis.

    Args:
        row: dict with key "concat" (str)

    Returns:
        dict: with keys "input_ids", "attention_mask", "token_type_ids"
              (each a list of 512 integers)
    """
    ## YOUR CODE HERE
    return row  # Replace this

# Subsample for training efficiency (do not change)

if __name__ == '__main__':
    train_mnli = mnli["train"].shuffle(seed=42).select(range(1000)).map(tokenize_function_nli)
    eval_mnli = mnli["validation_matched"].shuffle(seed=42).select(range(1000)).map(tokenize_function_nli)
    print(f"Train: {len(train_mnli)}, Eval: {len(eval_mnli)}")
    print(f"input_ids length: {len(train_mnli[0]['input_ids'])}")

# Sanity checks — 3.3
_r = tokenize_function_nli({"concat": "The cat sat. [SEP] An animal sat."})
assert "input_ids" in _r, "tokenize_function_nli must return 'input_ids'"
assert len(_r["input_ids"]) == 512, f"input_ids must be length 512, got {len(_r['input_ids'])}"
print("✓ 3.3 sanity checks passed")


### Problem 3.4: Define compute_metrics for NLI (7 points) [manually graded]

Same as Section 2: compute accuracy from logits and labels.

In [ ]:
def compute_metrics_nli(eval_pred):
    """Compute accuracy from model logits and true labels.

    Args:
        eval_pred: tuple (logits, labels)
            logits: np.ndarray of shape (n_examples, n_classes=3)
            labels: np.ndarray of shape (n,)

    Returns:
        dict: {"accuracy": float}
    """
    logits, labels = eval_pred
    ## YOUR CODE HERE
    return {"accuracy": None}  # Replace this

# Sanity checks — 3.4
import numpy as np
_logits = np.array([[0.9, 0.05, 0.05], [0.05, 0.9, 0.05], [0.05, 0.05, 0.9]])
_labels = np.array([0, 1, 2])
_r = compute_metrics_nli((_logits, _labels))
assert "accuracy" in _r, "compute_metrics_nli must return dict with 'accuracy' key"
assert abs(_r["accuracy"] - 1.0) < 1e-6, f"perfect predictions → accuracy=1.0, got {_r['accuracy']}"
print("✓ 3.4 sanity checks passed")


### Problem 3.5: Train and Evaluate (6 points) [autograded]

Your model should achieve at least **45% accuracy**. With default hyperparameters, expect ~55-65%.

In [ ]:
if __name__ == '__main__':
    ########## DO NOT MODIFY ##########
    training_args = TrainingArguments(
        output_dir="mnli-training",
        eval_strategy="epoch",
        logging_strategy="epoch",
        learning_rate=5e-05,
        num_train_epochs=3,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        report_to="none",
        seed=42,
    )

    trainer = Trainer(
        model=classification_head_bert,
        args=training_args,
        train_dataset=train_mnli,
        eval_dataset=eval_mnli,
        compute_metrics=compute_metrics_nli,
    )


In [ ]:
# Skip when imported as a module (slow training — ~5 min on GPU)
if __name__ == '__main__':
    # Train (~5 min on GPU)
    trainer.train()
    trainer.save_model("./mnliBERT")

In [ ]:
if __name__ == '__main__':
    # Sanity check
    mnliBERT = pipeline("text-classification", model="./mnliBERT", tokenizer=tokenizer, device=device_id)
    print(mnliBERT([
        "I just graduated college. [SEP] The person is a college graduate.",  # entailment
        "My name is John. [SEP] The sky is blue.",                            # neutral
        "Thank you for your help. [SEP] The person was not helped.",          # contradiction
    ]))


In [ ]:
# Load saved NLI eval results if available
eval_results_nli = None
try:
    eval_results_nli = _json.load(open('eval_results_nli.json'))
    print(f"NLI accuracy: {eval_results_nli['eval_accuracy']:.2%}")
except (FileNotFoundError, KeyError):
    pass

if __name__ == '__main__':
    eval_results_nli = trainer.evaluate()
    _json.dump({'accuracy': eval_results_nli['eval_accuracy']}, open('eval_results_nli.json', 'w'))
    print(f"NLI accuracy: {eval_results_nli['eval_accuracy']:.2%}")
    assert eval_results_nli["eval_accuracy"] >= 0.45, (
        f"Accuracy should be >= 45%, got {eval_results_nli['eval_accuracy']:.2%}."
    )


> **📥 Download for Gradescope!**
> After running the cells above, download **eval_results_nli.json** from the Colab file browser (folder icon on the left sidebar) and include it in your Gradescope submission.

---
## Section 4: Fine-Tuning BERT for Named Entity Recognition (25 points)

**Named Entity Recognition** (NER) is a **token classification** task: assign a label to each token in the sequence. The standard approach uses **BIO tagging**:

- `B-PER` = beginning of a person entity
- `I-PER` = inside/continuation of a person entity
- `B-ORG` = beginning of an organization entity; `I-ORG` = inside/continuation of an organization entity
- `B-LOC` = beginning of a location entity; `I-LOC` = inside/continuation of a location entity
- `B-MISC` = beginning of a miscellaneous entity; `I-MISC` = inside/continuation of a miscellaneous entity
- `O` = not an entity

**Example:**
```
Tokens:   Only  France  and  Britain  backed  Fischler's  proposal
BIO tags:  O    B-LOC    O   B-LOC     O      B-PER        O
```

**Connection to Lecture M07.2.3 and Chapter 10, §10.5:** Unlike sequence classification (which uses [CLS]), token classification uses the output vector for *each* token, passing it through a per-token classification head.

### The Subword Alignment Challenge

BERT uses **WordPiece tokenization**, which splits words into sub-tokens. For example, `Fischler` might become `["Fis", "##ch", "##ler"]`. The CoNLL dataset has one BIO tag per word, but after BERT tokenization we have multiple sub-tokens per word.

**Solution:** Assign the same BIO tag to all sub-tokens of a word.

We also need special handling for BERT's special tokens:
- `[CLS]`, `[SEP]`, `[PAD]` get label **-100**
- PyTorch's `CrossEntropyLoss` ignores positions with label -100 (via its `ignore_index` parameter)

In [ ]:
# The 9 BIO tag labels
label_names = {
    0: "O",
    1: "B-PER", 2: "I-PER",
    3: "B-ORG", 4: "I-ORG",
    5: "B-LOC", 6: "I-LOC",
    7: "B-MISC", 8: "I-MISC",
}
print("Label mapping:", label_names)

### Problem 4.1: Load BERT with a Token Classification Head (5 points) [autograded]

Use `AutoModelForTokenClassification` to load BERT with a token classification head. Set `num_labels=9` and `id2label=label_names`.

In [ ]:
if __name__ == '__main__':
    # Use AutoModelForTokenClassification.from_pretrained("bert-base-cased", ...)
    # with num_labels=9 and id2label=label_names so the model's config records the
    # BIO label names (handy when you inspect predictions later).
    ## YOUR CODE HERE
    token_classification_bert = None

    # Verify
    assert token_classification_bert is not None, "Load the model!"
    assert token_classification_bert.config.num_labels == 9, (
        f"Expected 9 labels, got {token_classification_bert.config.num_labels}"
    )
    print(f"Model loaded with {token_classification_bert.config.num_labels} labels")
    print(f"id2label: {token_classification_bert.config.id2label}")


In [ ]:
if __name__ == '__main__':
    # Load the CoNLL-2003 dataset
    conll = load_dataset("eriktks/conll2003")
    print(conll)
    print(f"\nExample:")
    print(f"  Tokens:   {conll['train'][12]['tokens']}")
    print(f"  NER tags: {conll['train'][12]['ner_tags']}")
    print(f"  Decoded:  {[label_names[t] for t in conll['train'][12]['ner_tags']]}")


### Problem 4.2: Tokenize and Redistribute BIO Tags (10 points) [autograded]

This is the most challenging function in this homework. You need to:

1. **Re-tokenize** each word with BERT's tokenizer (which may split words into sub-tokens)
2. **Assign the same BIO tag** to all sub-tokens of each word
3. **Add -100** for special tokens: [CLS] at the start, [SEP] at the end, [PAD] for padding
4. **Pad/truncate** both tokens and labels to exactly 512

#### Worked Example

```
Input:
  tokens:   ["Israel", "approves", "Arafat", "'s", "flight"]
  ner_tags: [5,        0,          1,        0,     0       ]
             B-LOC     O           B-PER     O      O

After BERT tokenization:
  sub_tokens: ["Israel", "approve", "##s", "Ara", "##fa", "##t", "'", "s", "flight"]
  sub_tags:   [5,        0,          0,     1,     1,      1,     0,   0,   0       ]

After adding special tokens and padding:
  final_tokens: ["[CLS]", "Israel", "approve", "##s", ..., "flight", "[SEP]", "[PAD]", ...]
  final_labels: [-100,     5,        0,          0,    ...,  0,       -100,    -100,    ...]
                 (512 total elements)
```

**Key insight:** When we tokenize individual words with `tokenizer.tokenize(word)`, we get the sub-tokens without special tokens. Then we use `tokenizer.convert_tokens_to_ids()` to get IDs, and manually add the [CLS] and [SEP] IDs.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")
MAX_LEN = tokenizer.model_max_length  # 512

# Useful token IDs
CLS_ID = tokenizer.cls_token_id  # 101
SEP_ID = tokenizer.sep_token_id  # 102
PAD_ID = tokenizer.pad_token_id  # 0

def tokenize_and_tag(row):
    """Re-tokenize with BERT's tokenizer and redistribute BIO tags.

    For each input word, BERT's WordPiece tokenizer may produce 1+ sub-tokens
    (e.g. "Fischler" -> ["Fis","##ch","##ler"]). Every sub-token of a word
    inherits that word's BIO tag. The special tokens that the model needs at
    the boundaries ([CLS], [SEP]) and any padding positions all get label
    `-100` so the loss ignores them.

    Args:
        row: dict with keys
            "tokens":   list of str (words)
            "ner_tags": list of int (one BIO tag id per word)

    Returns:
        dict with keys
            "input_ids":      list[int] of length 512 — BERT sub-token vocabulary IDs
                              (e.g. [101, 2142, 4107, 102, 0, 0, ...])
            "attention_mask": list[int] of length 512 — 1 for real tokens, 0 for [PAD]
            "labels":         list[int] of length 512 — NER tag ID per position
                              (-100 for [CLS], [SEP], and [PAD] so CrossEntropyLoss ignores them;
                               0=O, 1=B-PER, 2=I-PER, 3=B-ORG, 4=I-ORG,
                               5=B-LOC, 6=I-LOC, 7=B-MISC, 8=I-MISC for real tokens)

    Hint: the four phases of the function are
      1. expand each word into sub-tokens, replicating its tag once per piece;
      2. truncate to leave room for the special tokens (MAX_LEN - 2);
      3. wrap with [CLS]/[SEP] in input_ids and -100/-100 in labels;
      4. right-pad input_ids with PAD_ID, attention_mask with 0s, labels with -100.
    """
    ## YOUR CODE HERE
    return row

# Subsample and tokenize (do not change)

if __name__ == '__main__':
    train_conll = conll["train"].shuffle(seed=42).select(range(1000)).map(tokenize_and_tag)
    eval_conll = conll["validation"].shuffle(seed=42).select(range(1000)).map(tokenize_and_tag)
    print(f"Train: {len(train_conll)}, Eval: {len(eval_conll)}")
    print(f"input_ids length: {len(train_conll[0]['input_ids'])}")
    print(f"labels length: {len(train_conll[0]['labels'])}")


In [ ]:
if __name__ == '__main__':
    # Verify your implementation on a known example
    test_row = {
        "tokens": ["Only", "France", "and", "Britain", "backed", "Fischler", "proposal"],
        "ner_tags": [0, 5, 0, 5, 0, 1, 0]
    }
    result = tokenize_and_tag(test_row)

    # Print aligned tokens and labels
    tokens = tokenizer.convert_ids_to_tokens(result["input_ids"])
    for i in range(20):  # Show first 20 positions
        if tokens[i] == "[PAD]":
            break
        tag_name = label_names.get(result["labels"][i], "SPECIAL")
        print(f"  {i:3d}  {tokens[i]:15s}  {result['labels'][i]:4d}  ({tag_name})")


### Problem 4.3: Compute Macro-Averaged F1 Score (10 points) [autograded]

For NER evaluation, accuracy is misleading because most tokens are `O` (not entities). A model that predicts all `O` would have high accuracy but be useless.

Instead, we use **Macro-Averaged F1**: compute F1 for each class separately, then average them.

**Formula recap:**
- For each class $c$: Precision$_c$ = TP$_c$ / (TP$_c$ + FP$_c$), Recall$_c$ = TP$_c$ / (TP$_c$ + FN$_c$)
- F1$_c$ = 2 * P$_c$ * R$_c$ / (P$_c$ + R$_c$)   (0 if denominator is 0)
- Macro-F1 = mean of F1$_c$ over the classes that actually appear in the data

**Important:** Average over the classes that appear in `labels` or `preds` (i.e. the union of unique classes in the two lists). This matches the default behavior of `sklearn.metrics.f1_score(..., average='macro')`. If a class is not present in either, leave it out of the average rather than counting it as 0 — including absent classes would let a model with mediocre per-class F1 look much worse on small evaluation slices simply because rare classes never appear.

**Important:** Ignore positions where the label is `-100` (special tokens). These come from `[CLS]`, `[SEP]`, and padding.

In [ ]:
def calculate_macro_f1(preds, labels):
    """Compute macro-averaged F1 score for token classification.

    Args:
        preds:  list[list[int]], predicted label IDs per sequence.
        labels: list[list[int]], true label IDs per sequence (-100 = ignore).

    Returns:
        dict with keys "precision", "recall", "macro-f1" (all floats),
        each averaged over the classes that appear in `labels` or `preds`
        after dropping -100 positions.
    """
    label_map = {0: "O", 1: "B-PER", 2: "I-PER", 3: "B-ORG", 4: "I-ORG",
                 5: "B-LOC", 6: "I-LOC", 7: "B-MISC", 8: "I-MISC"}

    # Step 1: Flatten and filter out -100 positions
    all_preds = []
    all_labels = []
    for pred_seq, label_seq in zip(preds, labels):
        for p, l in zip(pred_seq, label_seq):
            if l in label_map:
                all_preds.append(p)
                all_labels.append(l)

    ## YOUR CODE HERE
    # Step 2: Determine the classes to average over (union of unique classes
    #         present in all_preds and all_labels).
    # Step 3: For each such class compute TP / FP / FN, then precision /
    #         recall / F1 (0 when the denominator is 0).
    # Step 4: Return the mean of each metric over the classes from step 2.

    return {"precision": None, "recall": None, "macro-f1": None}

In [ ]:
if __name__ == '__main__':
    # Debug: compare your implementation with sklearn
    from sklearn.metrics import f1_score

    for trial in range(5):
        rand_preds = np.random.randint(0, 9, 50).tolist()
        rand_labels = np.random.randint(0, 9, 50).tolist()
        sklearn_f1 = f1_score(rand_labels, rand_preds, average="macro")
        your_f1 = calculate_macro_f1([rand_preds], [rand_labels])["macro-f1"]
        match = "OK" if abs(sklearn_f1 - your_f1) < 0.001 else "MISMATCH"
        print(f"  Trial {trial}: sklearn={sklearn_f1:.4f}, yours={your_f1:.4f} [{match}]")


In [ ]:
# Wrap for the Trainer
def compute_metrics_ner(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=2)
    return calculate_macro_f1(preds.tolist(), labels.tolist())

In [ ]:
if __name__ == '__main__':
    ########## DO NOT MODIFY ##########
    training_args = TrainingArguments(
        output_dir="conll-training",
        eval_strategy="epoch",
        logging_strategy="epoch",
        learning_rate=5e-05,
        num_train_epochs=3,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        report_to="none",
        seed=42,
    )

    trainer = Trainer(
        model=token_classification_bert,
        args=training_args,
        train_dataset=train_conll,
        eval_dataset=eval_conll,
        compute_metrics=compute_metrics_ner,
    )


In [ ]:
# Skip when imported as a module (slow training — ~5 min on GPU)
if __name__ == '__main__':
    # Train (~5 min on GPU)
    trainer.train()
    trainer.save_model("./nerBERT")

In [ ]:
if __name__ == '__main__':
    # Sanity check
    nerBERT = pipeline("token-classification", model="./nerBERT", tokenizer=tokenizer, device=device_id)
    results = nerBERT(
        ["Chris Callison-Burch is a professor at the University of Pennsylvania in Philadelphia"],
        aggregation_strategy="simple"
    )
    for entity in results[0]:
        print(f"  {entity['word']:30s}  {entity['entity_group']:10s}  {entity['score']:.3f}")


In [ ]:
# Load saved NER eval results if available
eval_results_ner = None
try:
    eval_results_ner = _json.load(open('eval_results_ner.json'))
    print(f"Overall F1 (entity-level): {eval_results_ner['overall_f1']:.4f}")
except (FileNotFoundError, KeyError):
    pass

if __name__ == '__main__':
    # SeqEval evaluates at the entity level (stricter than token-level macro-F1).
    # Your model should achieve overall_f1 >= 0.70.
    import evaluate
    from evaluate import evaluator

    task_evaluator = evaluator("token-classification")
    eval_results_ner = task_evaluator.compute(
        model_or_pipeline=nerBERT,
        data=eval_conll,
        input_column="tokens",
        label_column="ner_tags",
        tokenizer=tokenizer,
        metric=evaluate.load("seqeval"),
    )
    _json.dump(
        {k: v for k, v in eval_results_ner.items() if isinstance(v, (int, float, str))},
        open('eval_results_ner.json', 'w'),
    )
    print(f"Overall F1 (entity-level): {eval_results_ner['overall_f1']:.4f}")
    for key in ["overall_precision", "overall_recall", "overall_f1", "overall_accuracy"]:
        print(f"  {key}: {eval_results_ner[key]:.4f}")


> **📥 Download for Gradescope!**
> After running the cells above, download **eval_results_ner.json** from the Colab file browser (folder icon on the left sidebar) and include it in your Gradescope submission.

---
## Section 5: Analysis & Ablation (15 points)

In Sections 2-4, you fine-tuned BERT by following a recipe: load model, tokenize data, train. But did you develop intuition for *why* this works or *when* it fails? This section addresses both questions.

### 5.1 Zero-Shot vs Fine-Tuned Comparison (5 points) [WRITEUP — manually graded]

**[WRITEUP] Answer 5.1:** Pick 10 specific Yelp reviews from the eval set. Run each through both:
- Your **zero-shot** approach (Section 1 mask-filling)
- Your **fine-tuned** sentiment classifier (Section 2)

Report a table showing each review's true label, zero-shot prediction, and fine-tuned prediction. Then answer: on which examples does zero-shot already work? On which does fine-tuning help most? What types of reviews does fine-tuning fix?

In [ ]:
## YOUR CODE HERE — compare zero-shot vs fine-tuned on 10 examples
# Pick 10 reviews from eval_data (Section 2)
# For each, get:
#   - true label (eval_data[i]["label"] + 1 to get 1-5 scale)
#   - zero-shot prediction (using bert_model mask-filling from Section 1)
#   - fine-tuned prediction (using yelpBERT pipeline from Section 2)
if __name__ == '__main__':
    pass

### 5.2 What Does Pre-training Buy You? (6 points code + 4 points writeup) [manually graded]

Fine-tuning works well because BERT was pre-trained on enormous amounts of text. But how much does pre-training actually help? Let's find out by **ablating** it: training the same architecture with **randomly initialized weights** (no pre-training).

**Instructions:**
1. Load a fresh BERT model with **randomly initialized weights** (not pre-trained)
2. Fine-tune it on the same 1000 Yelp sentiment examples with the same hyperparameters
3. Compare its accuracy to your pre-trained fine-tuned model

**To load BERT with random weights:**
```python
from transformers import BertConfig, BertForSequenceClassification

config = BertConfig.from_pretrained("bert-base-cased", num_labels=5)
random_bert = BertForSequenceClassification(config)  # Random weights!
# vs: pretrained_bert = AutoModelForSequenceClassification.from_pretrained("bert-base-cased", num_labels=5)
```

In [ ]:
if __name__ == '__main__':
    ## YOUR CODE HERE — Ablation: train BERT with random weights
    #
    # from transformers import BertConfig, BertForSequenceClassification
    #
    # config = BertConfig.from_pretrained("bert-base-cased", num_labels=5)
    # random_bert = BertForSequenceClassification(config)
    #
    # # Use the same TrainingArguments, train_data, eval_data, and compute_metrics
    # # from Section 2
    # trainer_random = Trainer(
    #     model=random_bert,
    #     args=TrainingArguments(
    #         output_dir="yelp-random",
    #         eval_strategy="epoch",
    #         logging_strategy="epoch",
    #         learning_rate=5e-05,
    #         num_train_epochs=3,
    #         per_device_train_batch_size=8,
    #         per_device_eval_batch_size=8,
    #         report_to="none",
    #         seed=42,
    #     ),
    #     train_dataset=train_data,
    #     eval_dataset=eval_data,
    #     compute_metrics=compute_metrics,
    # )
    #
    # trainer_random.train()
    # random_results = trainer_random.evaluate()
    #
    # print(f"Pre-trained BERT accuracy: {sentiment_eval_results['eval_accuracy']:.2%}")
    # print(f"Random init BERT accuracy: {random_results['eval_accuracy']:.2%}")
    # print(f"Difference (value of pre-training): {sentiment_eval_results['eval_accuracy'] - random_results['eval_accuracy']:+.2%}")
    pass

**[WRITEUP] Answer 5.2 (4 points):**

1. How much does pre-training improve accuracy over random initialization? Report both numbers.
2. Why is the gap so large with only 1000 training examples? What knowledge does pre-training provide that can't be learned from 1000 labeled reviews?
3. Would you expect this gap to shrink if you had 100,000 training examples? Why or why not?

---
## Submission Checklist

Before submitting to Gradescope, verify:

- [ ] All code cells run without errors (Kernel > Restart & Run All)
- [ ] Section 1: `concatenate_mask`, `filter_for_max_length`, `apply_bert` work correctly
- [ ] Section 2: Sentiment accuracy >= 45%
- [ ] Section 3: NLI accuracy >= 45%
- [ ] Section 4: NER macro-F1 reasonable, `calculate_macro_f1` matches sklearn
- [ ] Section 5: Ablation completed (5.1 + 5.2)
- [ ] `report.pdf` contains:
  - **Answer 5.1:** Zero-shot vs fine-tuned comparison (10 examples)
  - **Answer 5.2:** Pre-training ablation results and analysis
- [ ] Exported `hw06.py`: *File → Download → Download .py* (in Colab)
- [ ] Files named exactly: `hw06.py`, `hw06_notebook.ipynb`, `report.pdf`
- [ ] **Downloaded** eval_results_sentiment.json, eval_results_nli.json, eval_results_ner.json from Colab (folder icon ↗ on left sidebar) and added to submission


## Gradescope Submission

Upload **all four files** to Gradescope:

| File | How to get it                                   |
|------|-------------------------------------------------|
| hw06.py | File → Download → Download .py                  |
| hw06_notebook.ipynb | Download the notebook, with outputs intact      |
| eval_results_sentiment.json | Auto-saved when you ran the sentiment eval cell |
| eval_results_nli.json | Auto-saved when you ran the NLI eval cell       |
| eval_results_ner.json | Auto-saved when you ran the NER eval cell       |

> **Note:** The three JSON files are created automatically when you run the evaluation cells above. In Colab, use the file browser (folder icon on the left) to download them.